# 🤖 Construire son premier Agent IA — eXaltemps Value

# <img src="images/value_bandeau.png" alt="eXaltemps Value" width="1200"/>


### 💡 Votre LLM sait parler. Apprenez-lui à agir.

Cet atelier vous guide **pas à pas** pour comprendre ce qu'est un agent IA, puis en construire un avec LangChain : **outils**, **RAG** et **monitoring**.

À l'issue de cet atelier, vous aurez un agent capable de :
- 🧠 Raisonner et décider
- 🔧 Appeler des APIs (météo, web...)
- 📚 Interroger vos documents (RAG)

---

#### 📋 Programme

| # | Partie |
|---|--------|
| 0 | Qu'est-ce qu'un agent ? |
| 1 | Setup & Configuration |
| 2 | Construire l'agent (outils + ReAct) |
| 3 | Augmentation RAG |
| 4 | Monitoring |
| 5 | Chat avec l'agent |

---

## ⚙️ Configuration locale

Exécutez les deux cellules ci-dessous pour installer les dépendances et connecter le LLM (OpenRouter). Une clé API dans .env suffit.

In [3]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import warnings
from getpass import getpass

# Masquer l'avertissement Pydantic V1 / Python 3.14 (incompatibilit? connue LangChain)
warnings.filterwarnings("ignore", message=".*Pydantic V1.*Python 3.14.*", category=UserWarning)

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

if not os.getenv("OPENROUTER_API_KEY"):
    api_key = getpass("OPENROUTER_API_KEY : " ).strip()
    if api_key:
        os.environ["OPENROUTER_API_KEY"] = api_key

if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError("OPENROUTER_API_KEY est requis pour ex?cuter l'atelier.")

llm = ChatOpenAI(
    model="anthropic/claude-sonnet-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0.2
)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
)
print("Provider : OpenRouter (cloud)")
print("LLM      : anthropic/claude-sonnet-4.5")
print("Embeddings: text-embedding-3-small")


Provider : OpenRouter (cloud)
LLM      : anthropic/claude-sonnet-4.5
Embeddings: text-embedding-3-small


---

## Partie 1 — Qu'est-ce qu'un Agent ? 🤖

<img src="images/partie1.png" alt="eXaltemps Value" width="800"/>

Un **LLM seul** ne peut pas : connaître l'heure, la météo en temps réel, lire vos docs internes, appeler une API.

**Un agent résout ça** 🎯 en donnant des **outils** au LLM et en le laissant décider quand les utiliser.

> *"Le chatbot répond. L'agent agit."*

# <img src="images/react.png" alt="eXaltemps Value" width="600"/>


**Boucle ReAct :** Réfléchir → Choisir un outil → Observer → Réfléchir → Répondre

| | Chatbot | Agent |
|---|---|---|
| Comportement | Texte → Texte | Réfléchit, agit, puis répond |
| Outils | ❌ | ✅ Météo, API, DB, fichiers… |
| Exemple | "Capitale de la France ?" → "Paris" | "Météo Paris ?" → appel API → "12°C" |

### 🧪 Démonstration — Le LLM sans outils

Avant de construire un agent, observons ce qu'un LLM seul ne peut pas faire. Exécutez la cellule ci-dessous.

In [5]:
# Testons le LLM directement — sans agent, sans outils
print("=== LLM SEUL (sans outils) ===\n")

questions = [
    "Quelle heure est-il exactement en ce moment ?",
    "Quelle température fait-il à Paris ?",
]

for question in questions:
    response = llm.invoke(question)
    print(f"❓ {question}")
    print(f"🤖 {response.content[:500]}")
    print()

print("─" * 60)
print("→ Le LLM hésite, invente ou avoue ses limites.")
print("→ Avec un agent et des outils, ces réponses seront précises.")

=== LLM SEUL (sans outils) ===

❓ Quelle heure est-il exactement en ce moment ?
🤖 Je n'ai pas accès à l'heure actuelle en temps réel. Je ne peux pas consulter d'horloge ou connaître le moment exact où vous lisez ce message.

Pour connaître l'heure exacte, je vous suggère de :
- Regarder l'horloge de votre appareil (téléphone, ordinateur, montre)
- Demander à votre assistant vocal (Siri, Google Assistant, Alexa)
- Consulter un site comme time.is

Puis-je vous aider avec autre chose ?

❓ Quelle température fait-il à Paris ?
🤖 Je n'ai pas accès aux données météorologiques en temps réel, donc je ne peux pas vous dire quelle température il fait actuellement à Paris.

Pour obtenir cette information, vous pouvez :
- Consulter un site météo comme Météo-France, Weather.com ou Google Météo
- Utiliser une application météo sur votre smartphone
- Rechercher "météo Paris" sur un moteur de recherche

Ces sources vous donneront la température actuelle ainsi que les prévisions pour les prochains jours

---
## Partie 2 — Construire l'agent

<img src=images/LLM.png alt="Code et programmation" width="700" style="border-radius: 6px;"/>

### Étape 1 — Les outils

Un outil = une **fonction Python** + une **docstring**. C'est la docstring que le LLM lit pour décider de l'utiliser.

In [6]:
from datetime import datetime
from langchain_core.tools import tool


@tool
def get_current_time() -> str:
    """Retourne la date et l'heure actuelles. Utiliser quand l'utilisateur demande la date ou l'heure."""
    now = datetime.now()
    return now.strftime("%A %d %B %Y, %H:%M:%S")

In [7]:
@tool
def get_weather(city: str) -> str:
    """Retourne la météo actuelle pour une ville donnée.
    Utiliser quand l'utilisateur demande le temps qu'il fait, la météo, la température.
    Exemples : 'Paris', 'Lyon', 'Marseille', 'Bordeaux'
    """
    import urllib.request
    import json
    from urllib.parse import quote
    try:
        city_encoded = quote(city.strip())
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city_encoded}&count=1"
        with urllib.request.urlopen(geo_url) as response:
            geo_data = json.loads(response.read().decode())
        if not geo_data.get("results"):
            return f"Ville '{city}' non trouvée."
        lat = geo_data["results"][0]["latitude"]
        lon = geo_data["results"][0]["longitude"]
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,relative_humidity_2m,weather_code"
        with urllib.request.urlopen(weather_url) as response:
            weather_data = json.loads(response.read().decode())
        current = weather_data["current"]
        temp = current["temperature_2m"]
        humidity = current["relative_humidity_2m"]
        code = current["weather_code"]
        weather_desc = {
            0: "dégagé", 1: "principalement dégagé", 2: "partiellement nuageux",
            3: "nuageux", 45: "brouillard", 48: "brouillard givrant",
            51: "bruine", 61: "pluie légère", 63: "pluie", 65: "forte pluie",
            71: "neige légère", 73: "neige", 75: "forte neige"
        }
        desc = weather_desc.get(code, f"conditions météo (code {code})")
        return f"Météo à {city} : {temp}°C, {desc}, humidité {humidity}%"
    except Exception as e:
        return f"Erreur météo : {e}"

In [8]:
@tool
def search_web(query: str) -> str:
    """Recherche des informations sur le web à propos d'un sujet donné.
    Utiliser quand l'utilisateur pose une question d'actualité ou de culture générale
    que vous ne pouvez pas résoudre avec vos connaissances.
    """
    try:
        from ddgs import DDGS
        results = DDGS().text(query, region="fr-fr", max_results=5)
        if not results:
            return f"Aucun résultat trouvé pour '{query}'."
        parts = []
        for i, r in enumerate(results, 1):
            title = r.get("title", "")
            body = r.get("body", "")
            parts.append(f"{i}. {title}\n   {body}")
        return "Résultats de recherche :\n\n" + "\n\n".join(parts)
    except Exception as e:
        return f"Erreur lors de la recherche web : {e}"


tools = [get_current_time, get_weather, search_web]

for t in tools:
    print(f"🔧 {t.name}: {t.description[:200]}...")

🔧 get_current_time: Retourne la date et l'heure actuelles. Utiliser quand l'utilisateur demande la date ou l'heure....
🔧 get_weather: Retourne la météo actuelle pour une ville donnée.
    Utiliser quand l'utilisateur demande le temps qu'il fait, la météo, la température.
    Exemples : 'Paris', 'Lyon', 'Marseille', 'Bordeaux'...
🔧 search_web: Recherche des informations sur le web à propos d'un sujet donné.
    Utiliser quand l'utilisateur pose une question d'actualité ou de culture générale
    que vous ne pouvez pas résoudre avec vos conn...


### Étape 2 — Assembler l'agent

3 ingrédients : **prompt** (instructions) + **agent** (logique) + **executor** (boucle ReAct)

In [9]:
from langchain.agents import create_agent

SYSTEM_PROMPT = (
    "Tu es un assistant intelligent et serviable. "
    "Tu réponds toujours en français. "
    "Quand tu as besoin d'une information que tu ne connais pas "
    "(heure, météo, recherche web...), utilise les outils à ta disposition. "
    "Explique ton raisonnement de manière concise."
)

agent_executor = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    debug=False,
)

def _format_tool_call(tc):
    """Formate un tool_call pour affichage lisible."""
    args = tc.get("args") or {}
    args_str = ", ".join(f"{k}={repr(v)}" for k, v in args.items())
    return f"{tc.get('name', '?')}({args_str})"

def agent_invoke(executor, user_input: str, verbose: bool = True):
    """Appelle l'agent et retourne la réponse textuelle.
    Si verbose=True, affiche de façon lisible : raisonnement, appels d'outils, résultat."""
    from langchain_core.messages import AIMessage, ToolMessage
    final_answer = None
    for chunk in executor.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        stream_mode="updates",
    ):
        for node_name, node_output in chunk.items():
            if "messages" not in node_output:
                continue
            for msg in node_output["messages"]:
                if isinstance(msg, AIMessage):
                    if msg.content and msg.content.strip() and verbose:
                        print("💭 Raisonnement:", msg.content.strip(), end="\n\n")
                    if getattr(msg, "tool_calls", None) and verbose:
                        for tc in msg.tool_calls:
                            print("🔧", _format_tool_call(tc))
                    if not getattr(msg, "tool_calls", None):
                        final_answer = msg.content
                elif isinstance(msg, ToolMessage):
                    if verbose:
                        print("   →", msg.content, end="\n\n")
    if verbose and final_answer:
        print("📝 Réponse:\n", final_answer)
    return final_answer or ""

print("Agent créé avec succès !")

Agent créé avec succès !


### Étape 3 — Tester

Observez : raisonnement (💭), appels d'outils (🔧), réponse finale (📝).

In [10]:
# Test 1 : question avec outil (heure)
result = agent_invoke(agent_executor, "Quelle heure est-il ?")

🔧 get_current_time()
   → Wednesday 25 March 2026, 00:21:56

💭 Raisonnement: Il est actuellement **00h21** (minuit vingt-et-une minutes), dans la nuit du mercredi 25 mars 2026.

📝 Réponse:
 Il est actuellement **00h21** (minuit vingt-et-une minutes), dans la nuit du mercredi 25 mars 2026.


In [11]:
# Test 2 : météo
result = agent_invoke(agent_executor, "Quel temps fait-il à Paris ?")

🔧 get_weather(city='Paris')
   → Météo à Paris : 11.5°C, dégagé, humidité 55%

💭 Raisonnement: À Paris, il fait actuellement **11,5°C** avec un ciel **dégagé** et une humidité de **55%**. C'est une journée plutôt agréable avec un temps clair !

📝 Réponse:
 À Paris, il fait actuellement **11,5°C** avec un ciel **dégagé** et une humidité de **55%**. C'est une journée plutôt agréable avec un temps clair !


In [12]:
# Test 3 : deux outils en même temps (multi-step)
result = agent_invoke(agent_executor, "Quelle heure est-il et quel temps fait-il à Lyon ?")

💭 Raisonnement: Je vais vérifier l'heure actuelle et la météo à Lyon pour vous.

🔧 get_current_time()
🔧 get_weather(city='Lyon')
   → Wednesday 25 March 2026, 00:22:07

   → Météo à Lyon : 9.8°C, nuageux, humidité 66%

💭 Raisonnement: Il est actuellement **00h22** (mercredi 25 mars 2026).

À Lyon, la météo est la suivante :
- **Température** : 9,8°C
- **Conditions** : nuageux
- **Humidité** : 66%

C'est une nuit plutôt fraîche avec un ciel couvert !

📝 Réponse:
 Il est actuellement **00h22** (mercredi 25 mars 2026).

À Lyon, la météo est la suivante :
- **Température** : 9,8°C
- **Conditions** : nuageux
- **Humidité** : 66%

C'est une nuit plutôt fraîche avec un ciel couvert !


In [13]:
# Test 4 : sans outil — l'agent doit répondre directement
result = agent_invoke(agent_executor, "Quelle est la capitale du Japon ?")

💭 Raisonnement: La capitale du Japon est **Tokyo** (東京, Tōkyō).

Tokyo est la plus grande métropole du Japon et l'une des villes les plus peuplées au monde. Elle est le centre politique, économique et culturel du pays, abritant le Palais impérial (résidence de l'empereur du Japon) ainsi que le siège du gouvernement japonais.

📝 Réponse:
 La capitale du Japon est **Tokyo** (東京, Tōkyō).

Tokyo est la plus grande métropole du Japon et l'une des villes les plus peuplées au monde. Elle est le centre politique, économique et culturel du pays, abritant le Palais impérial (résidence de l'empereur du Japon) ainsi que le siège du gouvernement japonais.


---
## Partie 3 — Ajouter du RAG

<img src=images/RAG.png alt="Documents et knowledge base" width="700" style="border-radius: 6px;"/>

**RAG** = Retrieval-Augmented Generation : au lieu de tout mettre dans le prompt, on **cherche les passages pertinents** dans une base documentaire et on les injecte au LLM.

<img src=images/embeddings.png alt="Documents et knowledge base" width="700" style="border-radius: 6px;"/>


**Étape 1 :** Charger + découper le document en chunks

In [14]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/exalt_knowledge_base.md", encoding="utf-8")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "],
)
chunks = text_splitter.split_documents(documents)

print(f"Document chargé : {len(documents)} fichier(s)")
print(f"Découpé en {len(chunks)} chunks")
print(f"\nExemple de chunk :\n{'='*50}")
print(chunks[0])

Document chargé : 1 fichier(s)
Découpé en 18 chunks

Exemple de chunk :
page_content='# Base de connaissances : Politique de conges eXalt

Absences
🦊 Tu saisis dans Boondmanager tes congés payés ou RTT avant leur prise effective et avant de saisir ton temps de travail. 

Quelques règles importantes :' metadata={'source': 'data/exalt_knowledge_base.md'}


**Étape 2 :** Créer la base vectorielle 

In [15]:
from langchain_community.vectorstores import FAISS

# FAISS 
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

print(f"Base vectorielle créée avec {len(chunks)} documents")

# Test rapide : recherche sémantique
results = vectorstore.similarity_search("RTT", k=2)
print(f"\nRecherche 'RTT' → {len(results)} résultats :")
for i, doc in enumerate(results):
    print(f"\n--- Résultat {i+1} ---")
    print(doc)

Base vectorielle créée avec 18 documents

Recherche 'RTT' → 2 résultats :

--- Résultat 1 ---
page_content='Les RTT employeurs sont intégrés au sein de ton compteur global de RTT.
Le cercle RH s’occupe de les indiquer sur BoondManager. Le nombre de jours de RTT employeurs est communiqué en amont et dans le respect du maximum légal (50% du nombre de jours de RTT cumulés). 
Si pour un besoin au sein de ta mission tu dois travailler ce jour là, il est nécessaire de prévenir ton DSM et d'envoyer un e-mail au cercle RH de ton entité.' metadata={'source': 'data/exalt_knowledge_base.md'}

--- Résultat 2 ---
page_content='Le nombre de RTT varie chaque année et se définit au regard du nombre de jours fériés qui tombent en jour ouvré.​​​​​​​' metadata={'source': 'data/exalt_knowledge_base.md'}


**Étape 3 :** Transformer la recherche vectorielle en outil `@tool` pour l'agent

In [16]:
@tool
def search_knowledge_base(query: str) -> str:
    """Recherche dans la base de connaissances interne d'eXalt.
    Utiliser pour toute question sur l'entreprise eXalt : congés, RTT, absences,...
    """
    results = vectorstore.similarity_search(query, k=3)
    if not results:
        return "Aucune information trouvée dans la base de connaissances."
    context = "\n\n".join([doc.page_content for doc in results])
    return f"Informations trouvées dans la base de connaissances :\n\n{context}"


# Recréer l'agent avec le nouvel outil (sans mémoire — pour les tests)
tools_with_rag = [get_current_time, get_weather, search_web, search_knowledge_base]

agent_executor_rag = create_agent(
    model=llm,
    tools=tools_with_rag,
    system_prompt=SYSTEM_PROMPT,
    debug=False,
)

print("Agent RAG créé avec", len(tools_with_rag), "outils :")
for t in tools_with_rag:
    print(f"  - {t.name}")

Agent RAG créé avec 4 outils :
  - get_current_time
  - get_weather
  - search_web
  - search_knowledge_base


**Étape 4 :** Vérifier la structure de l'agent

Copie colle la sortie de cette cellule dans Think Agent !

In [17]:
# Architecture de l'agent RAG : résumé + graphe  (nœuds / transitions)
print("Architecture de l'agent RAG")
print("=" * 60)
model_id = getattr(llm, "model_name", None) or getattr(llm, "model", None) or str(llm)
print(f"LLM : {model_id}")
print("\nOutils :")
for t in tools_with_rag:
    desc = (t.description or "").strip().split("\n")[0] or "(pas de description)"
    print(f"  • {t.name} — {desc}")

print("\nBase vectorielle (FAISS) — utilisée par search_knowledge_base :")
idx = vectorstore.index
print(f"  • Vecteurs indexés : {idx.ntotal}")
print(f"  • Dimension des vecteurs : {idx.d}")
print(f"  • Type d'index : {type(idx).__name__}")
emb_label = getattr(embeddings, "model", None) or getattr(embeddings, "model_name", None)
if emb_label:
    print(f"  • Modèle d'embedding : {emb_label}")
print("  • Récupération dans l'outil RAG : k = 3 passages par requête")


Architecture de l'agent RAG
LLM : anthropic/claude-sonnet-4.5

Outils :
  • get_current_time — Retourne la date et l'heure actuelles. Utiliser quand l'utilisateur demande la date ou l'heure.
  • get_weather — Retourne la météo actuelle pour une ville donnée.
  • search_web — Recherche des informations sur le web à propos d'un sujet donné.
  • search_knowledge_base — Recherche dans la base de connaissances interne d'eXalt.

Base vectorielle (FAISS) — utilisée par search_knowledge_base :
  • Vecteurs indexés : 18
  • Dimension des vecteurs : 1536
  • Type d'index : IndexFlatL2
  • Modèle d'embedding : text-embedding-3-small
  • Récupération dans l'outil RAG : k = 3 passages par requête


**Étape 5 :** Tester l'agent RAG

In [ ]:
# Test 1 : question dans les docs
result = agent_invoke(agent_executor_rag, "Combien de jours de RTT par an ai-je droit chez eXalt ?")

In [ ]:
# Test 2 : formation
result = agent_invoke(agent_executor_rag, "Comment fonctionne les arrêts maladie ?")

In [ ]:
# Test 3 : RAG + météo (deux outils combinés)
result = agent_invoke(agent_executor_rag, "Combien de jours de congés + RTT ai-je au total par an ? Et quel temps fait-il à Paris ?")

---
## Partie 4 — Monitoring

<img src=images/langsmith.svg alt="Dashboard et analytics" width="1000" style="border-radius: 6px;"/>

**LangSmith** trace chaque appel LLM + outil (latence, tokens, coût). Décommentez les lignes ci-dessous pour activer.


In [ ]:
import os

langsmith_tracing = os.environ.get("LANGSMITH_TRACING", "false").lower() == "true"
langsmith_project = os.environ.get("LANGSMITH_PROJECT") or os.environ.get("LANGCHAIN_PROJECT", "default")

if langsmith_tracing:
    print("LangSmith actif ! Les traces sont visibles sur https://smith.langchain.com/")
    print(f"Projet : {langsmith_project}")
else:
    print("LangSmith désactivé.")


---
## Partie 5 — Chat avec l'agent

Discutez avec l'agent RAG. Une **nouvelle instance** est créée ici avec une mémoire (checkpointer) pour conserver le contexte. Vous pouvez faire des suivis, des précisions, poser des questions liées aux réponses précédentes.

Exécutez les deux cellules ci-dessous (création de l'agent, puis chat). Tapez `quit` ou `exit` pour terminer le chat.

In [ ]:
# Créer une instance de l'agent avec mémoire pour le chat
from langgraph.checkpoint.memory import MemorySaver

chat_memory = MemorySaver()
agent_executor_chat = create_agent(
    model=llm,
    tools=tools_with_rag,
    system_prompt=SYSTEM_PROMPT,
    debug=False,
    checkpointer=chat_memory,
)
print("Agent chat (avec mémoire) créé.")

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

# Identifiant de la conversation (changez pour démarrer une nouvelle discussion)
THREAD_ID = "chat-session-1"
config = {"configurable": {"thread_id": THREAD_ID}}

def chat_with_agent(executor, verbose: bool = False):
    """Boucle de chat interactive. L'agent conserve le contexte."""
    print("💬 Chat avec l'agent (mémoire activée). Tapez 'quit' ou 'exit' pour terminer.\n")
    while True:
        user_input = input("Vous : ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "q"):
            print("À bientôt !")
            break
        # Invoke avec le thread_id — le checkpointer charge l'historique
        result = executor.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config,
        )
        # Dernier message de l'agent (après éventuels tool calls)
        messages = result.get("messages", [])
        last_ai_msg = None
        for m in reversed(messages):
            if isinstance(m, AIMessage) and m.content:
                last_ai_msg = m.content
                break
        response = last_ai_msg or "(Pas de réponse textuelle)"
        print(f"\n🤖 Agent : {response}\n")

chat_with_agent(agent_executor_chat, verbose=False)

---

**Ressources :** [LangChain docs](https://python.langchain.com/docs/) · [LangGraph](https://langchain-ai.github.io/langgraph/) · [Cours DeepLearning.AI](https://www.deeplearning.ai/short-courses/ai-agents-in-langgraph/)

*Atelier eXaltemps — Construire son premier Agent IA — 2026*